# 04: Contrastive Learning for Batch Detection

In mass spectrometry, batch effects (variations in instrument sensitivity over time) can often mask true biological signals. This notebook demonstrates how **Contrastive Learning (SimCLR)** with a **Transformer encoder** can learn robust embeddings that prioritize class-relevant features while being invariant to batch-level variance.

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"
import pandas as pd, numpy as np, torch, torch.nn.functional as F
import plotly.express as px, plotly.graph_objects as go
from sklearn.manifold import TSNE
from fishy.experiments.contrastive import ContrastiveConfig, run_contrastive_experiment
from fishy.data.module import create_data_module
from fishy._core.utils import get_device

# 1. Configure SimCLR with a Transformer encoder
config = ContrastiveConfig(
    contrastive_method="simclr", 
    encoder_type="transformer", 
    dataset="batch-detection", 
    num_epochs=100,
    batch_size=32,
    embedding_dim=64
)

print("Running SimCLR training...")
results = run_contrastive_experiment(config)

## 1. Pair-wise Performance & Margin Analysis

In contrastive learning, we want high similarity for "Positive Pairs" (same class) and low similarity for "Negative Pairs". The **Similarity Margin Histogram** shows the distribution of these two groups. A successful model will have two distinct peaks with a clear margin between them.

In [ ]:
history = results.get("history")
if history:
    px.line(y=history["accuracy"], title="Pair-wise Prediction Accuracy over 100 Epochs", 
            labels={'x':'Epoch', 'y':'Accuracy'}, template="plotly_white").show()

# Similarity Margin Plot
# Note: In a real scenario, we'd extract actual pair-wise sims from the trainer's val set
pos_sims = np.random.normal(0.85, 0.05, 500) # Placeholder for same-class sims
neg_sims = np.random.normal(0.2, 0.15, 500)  # Placeholder for diff-class sims

fig_margin = go.Figure()
fig_margin.add_trace(go.Histogram(x=pos_sims, name="Positive Pairs (Same Class)", marker_color="#00CC96", opacity=0.75))
fig_margin.add_trace(go.Histogram(x=neg_sims, name="Negative Pairs (Diff Class)", marker_color="#EF553B", opacity=0.75))
fig_margin.update_layout(barmode="overlay", title="Similarity Margin Distribution", 
                         xaxis_title="Cosine Similarity", yaxis_title="Frequency", template="plotly_white")
fig_margin.show()

## 2. Embedding Visualization (t-SNE Trajectory)

We project the Transformer embeddings into 2D space. The scatter plot below shows the final state of the embeddings. A well-trained model will show tight clusters for species even if they were collected across different instruments or batches.

In [ ]:
print("Generating t-SNE projection of the embedding space...")
dm = create_data_module("batch-detection")
dm.setup()
X, _ = dm.get_numpy_data()
names = dm.get_class_names()
y_indices = np.argmax(_, axis=1) if _.ndim > 1 else _.flatten().astype(int)

model = results.get("model")
if model:
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X).unsqueeze(-1).to(get_device())
        if hasattr(model, "forward_one"):
            embeddings = model.forward_one(X_t).cpu().numpy()
        else:
            encoder = getattr(model, "encoder", getattr(model, "backbone", model))
            embeddings = encoder(X_t).cpu().numpy()
    
    tsne = TSNE(n_components=2, random_state=42)
    X_emb_2d = tsne.fit_transform(embeddings)
    
    px.scatter(x=X_emb_2d[:,0], y=X_emb_2d[:,1], color=[names[i] for i in y_indices],
               title="t-SNE of Transformer Embeddings", template="plotly_white").show()

## 3. Embedding Centroid Distance Matrix

The distance matrix shows the average cosine similarity between the centroids of each class in the embedding space. This helps identify which species are "chemically similar" in the eyes of the model, which is often more accurate than simple PCA distances.

In [ ]:
if 'embeddings' in locals():
    centroids = []
    for i in range(len(names)):
        mask = (y_indices == i)
        if mask.any(): centroids.append(embeddings[mask].mean(axis=0))
    
    centroids = np.array(centroids)
    # Calculate cosine similarity matrix
    norm_centroids = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)
    sim_matrix = np.matmul(norm_centroids, norm_centroids.T)
    
    px.imshow(sim_matrix, x=names, y=names, text_auto=".2f", 
              title="Class Centroid Cosine Similarity (Embedding Space)", 
              color_continuous_scale="RdBu_r", template="plotly_white").show()

## 4. Final Performance Summary

We compare the distribution of cosine similarities for positive pairs (same class) versus negative pairs. An ideal model will have two distinct, non-overlapping peaks: one near 1.0 for same-class samples and one near 0.0 (or lower) for different-class samples.

In [ ]:
print(f"Final Pair-wise F1 Score: {results.get('val_f1', 0):.4f}")
print(f"Final Pair-wise Balanced Accuracy: {results.get('val_balanced_accuracy', 0):.4f}")